In [1]:
import openmm as mm
from openmm import app, unit
from openmm.app.gromacsgrofile import GromacsGroFile

import martini_openmm as martini

import numpy as np
import sys

from openmmnapshift.utils import RESIDUE_TYPES, get_napshift_force

# Martini

In [5]:
temperature = 298*unit.kelvin
timestep = 10*unit.femtosecond
pressure = 1*unit.bar

max_K = 25
K_gradient = 0.001

report_interval = 1000

conf = GromacsGroFile(f"Data/2ND3/system.gro")
box_vectors = conf.getPeriodicBoxVectors()
topfile = martini.MartiniTopFile(
    f"Data/2ND3/system.top",
    periodicBoxVectors=box_vectors,
    epsilon_r=15,
)
topology = topfile.topology
system = topfile.create_system(nonbonded_cutoff=1.1 * unit.nanometer)
barostat = mm.MonteCarloBarostat(pressure, temperature)
system.addForce(barostat)

napshift_force = get_napshift_force(topology, 'Data/2ND3/26046_CS_0.txt', model_type='martini')
napshift_force.setUsesPeriodicBoundaryConditions(True)
system.addForce(napshift_force)
print("Successfully added NapShift Force")

integrator = mm.LangevinMiddleIntegrator(temperature, 0.01/unit.picosecond, timestep)
platform = mm.Platform.getPlatformByName('CUDA')
simulation = app.Simulation(topology, system, integrator, platform, {"Precision" : "mixed", 'DeviceIndex' : "0"})
simulation.context.setPositions(conf.getPositions())
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)

xtc_reporter = app.XTCReporter('Data/2ND3/output.xtc', report_interval, append=False, enforcePeriodicBox=True, atomSubset=[atom.index for atom in topology.atoms() if atom.residue.name in RESIDUE_TYPES.keys()])
state_data_reporter = app.StateDataReporter(sys.stdout, report_interval, step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True)
simulation.reporters.append(xtc_reporter)
simulation.reporters.append(state_data_reporter)

print(f"Simulating without CS restraints")
simulation.step(100000)

warmup_steps = int(np.floor(max_K/K_gradient))
print(f"Warming up CS restraints for {len(range(warmup_steps))} steps")
for i in range(warmup_steps):
    simulation.step(1)
    simulation.context.setParameter('NapShift_K', (i*K_gradient))
    
print(f"Simulating with CS restraints")
simulation.step(100000)

/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105

Successfully added NapShift Force
Simulating without CS restraints
#"Step","Time (ps)","Potential Energy (kJ/mole)","Kinetic Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Box Volume (nm^3)","Speed (ns/day)"
1000,9.999999999999831,-48829.93811279074,3460.6655630849586,-45369.27254970578,173.4982898238806,166.9574428002589,0
2000,20.000000000000327,-48706.41593104205,3760.285237534219,-44946.13069350783,188.51953361843977,166.93700650676558,2.69e+03
3000,30.00000000000189,-48445.8043201526,3951.788570959069,-44494.01574919353,198.12043270536728,167.81615079747408,2.69e+03
4000,40.00000000000061,-48157.51439867992,3982.0654958595173,-44175.448902820404,199.63834727861806,167.60504114329206,2.7e+03
5000,49.99999999999862,-48078.191335008494,4207.496098107132,-43870.69523690136,210.94016863377465,168.01733773002107,2.69e+03
6000,59.99999999999663,-47921.11427237244,4366.499134704391,-43554.61513766805,218.91168579529906,169.46948635536478,2.69e+03
7000,69.9999999999989,-4791

# All Atom

In [8]:
temperature = 298*unit.kelvin
timestep = 2*unit.femtosecond
pressure = 1*unit.bar
collision_frequency = 1/unit.picosecond

max_K = 25
K_gradient = 0.001

report_interval = 1000

gro = app.GromacsGroFile(f'Data/ShortHelix/system.gro')
top = app.GromacsTopFile(f'Data/ShortHelix/system.top', periodicBoxVectors=gro.getPeriodicBoxVectors(),
        includeDir=f'Data/GromacsForceFields/top')
system = top.createSystem(nonbondedMethod=app.PME, nonbondedCutoff=1*unit.nanometer,
        constraints=app.HBonds)

system.addForce(mm.AndersenThermostat(temperature, collision_frequency))
system.addForce(mm.MonteCarloBarostat(pressure, temperature))

napshift_force = get_napshift_force(top.topology, 'Data/ShortHelix/CS.txt', model_type='all_atom')
napshift_force.setUsesPeriodicBoundaryConditions(True)
system.addForce(napshift_force)

integrator = mm.VerletIntegrator(timestep)
platform = mm.Platform.getPlatformByName("CUDA")
simulation = app.Simulation(top.topology, system, integrator, platform, {"Precision" : "mixed", 'DeviceIndex' : "0"})
simulation.context.setPositions(gro.positions)

xtc_reporter = app.XTCReporter('Data/ShortHelix/output.xtc', report_interval, append=False, enforcePeriodicBox=True, atomSubset=[atom.index for atom in top.topology.atoms() if atom.residue.name in RESIDUE_TYPES.keys()])
state_data_reporter = app.StateDataReporter(sys.stdout, report_interval, step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True)
simulation.reporters.append(xtc_reporter)
simulation.reporters.append(state_data_reporter)

print("Minimizing Energy")
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)

warmup_steps = int(np.floor(max_K/K_gradient))
print(f"Warming up CS restraints for {len(range(warmup_steps))} steps")
for i in range(warmup_steps):
    simulation.step(1)
    simulation.context.setParameter('NapShift_K', (i*K_gradient))
    
print(f"Simulating with CS restraints")
simulation.step(100000)

/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105

Minimizing Energy
Warming up CS restraints for 25000 steps
#"Step","Time (ps)","Potential Energy (kJ/mole)","Kinetic Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Box Volume (nm^3)","Speed (ns/day)"
1000,2.0000000000000013,-490737.028302548,57342.86620931978,-433394.1620932282,217.47770008340078,316.6823925747623,0
2000,3.999999999999781,-476163.3354007425,65481.561943392764,-410681.77345734974,248.34439627301515,316.6107608883066,184
3000,5.999999999999561,-466506.4191557688,70180.70889860648,-396325.71025716234,266.1663110068081,315.57824062521814,183
4000,7.999999999999341,-458940.5134199094,73148.20631050973,-385792.3071093997,277.4207974809996,316.1034227319835,182
5000,10.000000000000009,-453205.86136650457,74268.03162033182,-378937.82974617276,281.6678302677163,316.86091108939996,182
6000,12.000000000000677,-451435.9205341127,76263.67462242996,-375172.2459116828,289.2364761861043,317.0172507402656,182
7000,14.000000000001345,-450098.3864227447,77594.62960792033,-

# CA-Only

In [29]:
import os
import sys
import numpy as np
import pandas as pd

def genParamsDH(temp,ionic):
    """ Debye-Huckel parameters. """

    kT = 8.3145*temp*1e-3
    # Calculate the prefactor for the Yukawa potential
    fepsw = lambda T : 5321/T+233.76-0.9297*T+0.1417*1e-2*T*T-0.8292*1e-6*T**3
    epsw = fepsw(temp)
    lB = 1.6021766**2/(4*np.pi*8.854188*epsw)*6.02214076*1000/kT
    yukawa_eps = lB*kT
    # Calculate the inverse of the Debye length
    yukawa_kappa = np.sqrt(8*np.pi*lB*ionic*6.02214076/10)
    return yukawa_eps, yukawa_kappa

def get_Ashbaugh_Hatch(lj_eps, cutoff, params, top, lambdas_column):
    energy_expression = 'select(step(r-2^(1/6)*s),4*eps*l*((s/r)^12-(s/r)^6-shift),4*eps*((s/r)^12-(s/r)^6-l*shift)+eps*(1-l))'
    ah = mm.CustomNonbondedForce(energy_expression + '; s=0.5*(s1+s2); l=0.5*(l1+l2); shift=(0.5*(s1+s2)/rc)^12-(0.5*(s1+s2)/rc)^6')
    ah.addGlobalParameter('eps', lj_eps * unit.kilojoules_per_mole)
    ah.addGlobalParameter('rc', float(cutoff) * unit.nanometer)
    ah.addPerParticleParameter('s')
    ah.addPerParticleParameter('l')

    for r in top.residues():
        ah.addParticle([params.loc[r.name].sigmas * unit.nanometer, params.loc[r.name][lambdas_column] * unit.dimensionless])
           
    ah.setNonbondedMethod(mm.CustomNonbondedForce.CutoffPeriodic)
    ah.setCutoffDistance(cutoff*unit.nanometer)
    ah.setForceGroup(0)
    return ah

def get_Yukawa(yukawa_kappa, yukawa_eps, params, top):
    yu = mm.CustomNonbondedForce('q*(exp(-kappa*r)/r-shift); q=q1*q2')
    yu.addGlobalParameter('kappa', yukawa_kappa / unit.nanometer)
    yu.addGlobalParameter('shift', np.exp(-yukawa_kappa * 4.0) / 4.0 / unit.nanometer)
    yu.addPerParticleParameter('q')

    for r in top.residues():
        #sqrt(eps)*sqrt(eps)=eps
        yu.addParticle([params.loc[r.name].q*np.sqrt(yukawa_eps) * unit.nanometer * unit.kilojoules_per_mole])

    yu.setNonbondedMethod(mm.CustomNonbondedForce.CutoffPeriodic)
    yu.setCutoffDistance(4*unit.nanometer)
    yu.setForceGroup(1)
    return yu

def add_bonds(top, bond_length, k_bond):
    harmonic_bond_force = mm.HarmonicBondForce()
    harmonic_bond_force.setUsesPeriodicBoundaryConditions(True)
    exclusions_1_2 = [] # for ah, yu etc.
    for chain in top.chains():
        atoms = [atom for atom in chain.atoms()]
        for i in range(len(chain)-1):
            harmonic_bond_force.addBond(atoms[i].index, atoms[i+1].index, bond_length*unit.nanometer, k_bond*unit.kilojoules_per_mole/(unit.nanometer**2))
            exclusions_1_2.append((atoms[i].index, atoms[i+1].index))
    return harmonic_bond_force, exclusions_1_2

def add_angles_restriction(top):
    restrict_angle_force = mm.CustomAngleForce("ReB_K/((sin(theta))^2)")
    restrict_angle_force.addGlobalParameter("ReB_K", 0)
    restrict_angle_force.setUsesPeriodicBoundaryConditions(True)
    exclusions_1_3 = [] # for ah, yu etc.
    for chain in top.chains():
        atoms = [atom for atom in chain.atoms()]
        for i in range(len(chain)-2):
            restrict_angle_force.addAngle(atoms[i].index, atoms[i+1].index, atoms[i+2].index)
            exclusions_1_3.append([atoms[i].index, atoms[i+1].index, atoms[i+2].index])
    return restrict_angle_force, exclusions_1_3

max_K = 25
K_gradient = 0.0001

report_interval = 1000

temperature = 298*unit.kelvin
salt_conc = 0.165
timestep = 10*unit.femtosecond

bond_length = 0.38
k_bond = 8033.0
eps_lj = 0.2 * 4.184 # kcal to kJ/mol
cutoff_lj = 2.2
yukawa_eps, yukawa_kappa = genParamsDH(temperature.value_in_unit(unit.kelvin), salt_conc)
CALVADOS_parameters = pd.read_csv('Data/CALVADOS_parameters.csv', index_col='three')

exclusions_1_2 = []
exclusions_1_3 = []
exclusions_1_4 = []

cg_pdb = app.PDBFile('Data/1DJF/CA.pdb')
top = cg_pdb.topology

system = mm.System()
for r in top.residues():
    system.addParticle(CALVADOS_parameters.loc[r.name].MW*unit.amu)
system.setDefaultPeriodicBoxVectors(np.array([10,0,0]), np.array([0,10,0]), np.array([0,0,10]))

ah = get_Ashbaugh_Hatch(eps_lj, cutoff_lj, CALVADOS_parameters, top, 'CALVADOS2')
yu = get_Yukawa(yukawa_eps, yukawa_kappa, CALVADOS_parameters, top)
hb, exclusions_1_2 = add_bonds(top, bond_length, k_bond)
system.addForce(hb)

for i, j in exclusions_1_2:
    ah.addExclusion(i,j)
    yu.addExclusion(i,j)

napshift_force = get_napshift_force(top, 'Data/1DJF/CS.txt', model_type='CA')
napshift_force.setUsesPeriodicBoundaryConditions(True)
system.addForce(napshift_force)

restricted_angles, _ = add_angles_restriction(top)
system.addForce(restricted_angles)

system.addForce(ah)
system.addForce(yu)

integrator = mm.LangevinMiddleIntegrator(temperature, 0.01/unit.picosecond, timestep)
platform = mm.Platform.getPlatformByName("CUDA")
simulation = app.Simulation(top, system, integrator, platform, {"Precision" : "mixed", 'DeviceIndex' : "0"})
simulation.context.setPositions(cg_pdb.getPositions())

xtc_reporter = app.XTCReporter('Data/1DJF/output.xtc', report_interval, append=False, enforcePeriodicBox=True)
state_data_reporter = app.StateDataReporter(sys.stdout, report_interval, step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True)
simulation.reporters.append(xtc_reporter)
simulation.reporters.append(state_data_reporter)
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)

print(f"Simulating without CS restraints")
simulation.step(100000)

warmup_steps = int(np.floor(max_K/K_gradient))
print(f"Warming up CS restraints for {len(range(warmup_steps))} steps")
for i in range(warmup_steps):
    simulation.step(1)
    simulation.context.setParameter('NapShift_K', (i*K_gradient))
    simulation.context.setParameter('ReB_K', (i*(1/warmup_steps)))
simulation.context.setParameter('ReB_K', 1)   

print(f"Simulating with CS restraints")
simulation.step(100000)

/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/Science/openmm-napshift/tutorials/napshift_cg_/src/random_coil/camcoil.py:105

Simulating without CS restraints
#"Step","Time (ps)","Potential Energy (kJ/mole)","Kinetic Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Box Volume (nm^3)","Speed (ns/day)"
1000,9.999999999999831,2.5788115004434076,33.89163797649066,36.470449476934064,181.16564958613432,1000.0,0
2000,20.000000000000327,1.5126159027968242,44.614526739012064,46.12714264180889,238.4841866674513,1000.0,4.36e+03
3000,30.00000000000189,3.5670186984352767,43.66615784965872,47.233176548093994,233.41473956648323,1000.0,4.35e+03
4000,40.00000000000061,-0.2161498221103102,50.95378528074525,50.73763545863494,272.3703001802943,1000.0,4.35e+03
5000,49.99999999999862,19.09896725847284,39.58032100737287,58.67928826584571,211.57415204016974,1000.0,4.35e+03
6000,59.99999999999663,4.352316773387429,55.814462421925455,60.166779195312884,298.35274848572044,1000.0,4.35e+03
7000,69.9999999999989,9.307284427370178,39.58050864749572,48.8877930748659,211.57515505881307,1000.0,4.35e+03
8000,80.00000000000402,8.08